# PointNet++ — Point Cloud Segmentation

Beginner-friendly notebook. Runs top to bottom.



In [ ]:
# ── Install (run once if needed) ─────────────────────────────────────────────
# pip install torch laspy open3d numpy scikit-learn joblib tqdm mlflow matplotlib
# For PTv1/PTv2/GNN also:
# pip install torch-scatter torch-cluster -f https://data.pyg.org/whl/torch-<VER>+<CUDA>.html

import os, glob, glob, copy, random, logging
import numpy as np
import torch
import torch.nn as nn
import open3d as o3d
import laspy
import joblib
import mlflow
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

# Use GPU if available, otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Running on: {DEVICE}")


In [ ]:
# ── Configuration — change values here, nowhere else ────────────────────────
CONFIG = {
    "train_dir"      : "data/train",      # labelled .las files
    "test_dir"       : "data/test",       # unlabelled .las files for inference
    "checkpoint_dir" : "checkpoints",     # where the best model is saved
    "num_classes"    : 2,                 # 0 = environment, 1 = wood powder
    "target_class"   : 1,                 # class we want to highlight (green)
    "val_ratio"      : 0.15,
    "test_ratio"     : 0.15,
    "seed"           : 42,
    # ── training ──
    "epochs"         : 100,
    "patience"       : 30,               # early stop if val mIoU doesn't improve
    "batch_size"     : 8,
    "num_points"     : 4096,             # points per training chunk
    "chunks_per_cloud": 4,
    "lr"             : 1e-3,
    "in_channels"    : 7,                # xyz + height + normals
    # ── visualization ──
    "max_vis_files"  : 3,                # how many test files to show in 3D
    # ── MLflow ──
    "mlflow_uri"     : "http://localhost:5000",
    "mlflow_experiment": "PointCloud_Segmentation",
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
NUM_CLASSES = CONFIG["num_classes"]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])


In [ ]:
# ── Point cloud loader ───────────────────────────────────────────────────────
def load_pointcloud(path):
    """Read a .las/.laz file. Returns (points[N,3], labels[N] or None)."""
    las = laspy.read(path)
    pts = np.column_stack([np.asarray(las.x),
                           np.asarray(las.y),
                           np.asarray(las.z)]).astype(np.float64)
    labels = None
    for key in ("classification", "label", "labels", "class"):
        if key in las.point_format.dimension_names:
            labels = np.asarray(getattr(las, key), dtype=np.int64)
            break
    return pts, labels

def list_files(folder):
    """Return sorted list of .las/.laz files in a folder."""
    files = []
    for ext in (".las", ".laz"):
        files += glob.glob(os.path.join(folder, "*" + ext))
    return sorted(files)

# ── Split labelled files into train / val / test ─────────────────────────────
all_files = list_files(CONFIG["train_dir"])
assert all_files, f"No .las files found in {CONFIG['train_dir']}"

rng   = np.random.RandomState(CONFIG["seed"])
order = rng.permutation(len(all_files))
n_val  = max(1, int(len(all_files) * CONFIG["val_ratio"]))
n_test = max(1, int(len(all_files) * CONFIG["test_ratio"]))

VAL_FILES   = [all_files[i] for i in order[:n_val]]
TEST_FILES  = [all_files[i] for i in order[n_val : n_val + n_test]]
TRAIN_FILES = [all_files[i] for i in order[n_val + n_test :]]
INFER_FILES = list_files(CONFIG["test_dir"])   # no labels, final inference only

log.info(f"train={len(TRAIN_FILES)}  val={len(VAL_FILES)}  "
         f"test={len(TEST_FILES)}  inference={len(INFER_FILES)}")


In [ ]:
# ── 7-channel feature computation ───────────────────────────────────────────
# Each point gets: normalized xyz (3) + height above floor (1) + surface normal (3)
def make_features(points):
    """Turn raw XYZ into 7-channel features. Returns float32 array (N,7)."""
    center = points.mean(axis=0, keepdims=True)
    scale  = max(np.linalg.norm(points - center, axis=1).max(), 1e-9)
    norm_xyz = ((points - center) / scale).astype(np.float32)

    z = points[:, 2]
    height = ((z - z.min()) / max(z.max() - z.min(), 1e-6)).astype(np.float32)

    # surface normals via Open3D
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(16))
    pcd.orient_normals_to_align_with_direction([0., 0., 1.])
    normals = np.asarray(pcd.normals, dtype=np.float32)

    return np.column_stack([norm_xyz, height[:, None], normals])

# ── Feature cache (compute once per file, reuse every epoch) ─────────────────
FEATURE_CACHE = {}

def get_features(path):
    """Return (features[N,7], labels[N]) for a file, cached after first call."""
    if path not in FEATURE_CACHE:
        pts, lbl = load_pointcloud(path)
        feat = make_features(pts)
        lbl  = np.zeros(len(pts), dtype=np.int64) if lbl is None else lbl
        lbl  = np.clip(lbl, 0, NUM_CLASSES - 1).astype(np.int64)
        FEATURE_CACHE[path] = (pts, feat, lbl)
    return FEATURE_CACHE[path]

log.info("Pre-computing features for train + val files (one-time cost)...")
for f in tqdm(TRAIN_FILES + VAL_FILES, desc="features"):
    get_features(f)
log.info("Done.")


In [ ]:
# ── Metrics ──────────────────────────────────────────────────────────────────
def compute_miou(true_labels, pred_labels, num_classes):
    """Compute mean Intersection-over-Union across all classes."""
    ious = []
    for c in range(num_classes):
        tp = int(((true_labels == c) & (pred_labels == c)).sum())
        fp = int(((true_labels != c) & (pred_labels == c)).sum())
        fn = int(((true_labels == c) & (pred_labels != c)).sum())
        if tp + fp + fn > 0:
            ious.append(tp / (tp + fp + fn))
    return float(np.mean(ious)) if ious else 0.0


In [ ]:
# ── MLflow setup ─────────────────────────────────────────────────────────────
try:
    mlflow.set_tracking_uri(CONFIG["mlflow_uri"])
    mlflow.set_experiment(CONFIG["mlflow_experiment"])
    MLFLOW_OK = True
    log.info(f"MLflow tracking: {CONFIG['mlflow_uri']}")
except Exception as e:
    MLFLOW_OK = False
    log.warning(f"MLflow not available ({e}) — training continues without logging")


## Model Architecture

In [ ]:
MODEL_NAME = "PointNet++"

from torch.utils.data import Dataset, DataLoader

def square_distance(a, b):
    return ((a[:,:,None,:] - b[:,None,:,:]) ** 2).sum(-1)

def index_points(p, idx):
    B = p.shape[0]
    bi = torch.arange(B, device=p.device).view(B,1,1).expand_as(idx)
    return p[bi, idx]

def farthest_point_sample(xyz, npoint):
    B, N, _ = xyz.shape
    idx  = torch.zeros(B, npoint, dtype=torch.long, device=xyz.device)
    dist = torch.full((B, N), 1e10, device=xyz.device)
    sel  = torch.randint(0, N, (B,), device=xyz.device)
    for i in range(npoint):
        idx[:, i] = sel
        d = ((xyz - xyz[torch.arange(B), sel].unsqueeze(1)) ** 2).sum(-1)
        dist = torch.minimum(dist, d)
        sel  = dist.argmax(-1)
    return idx

def ball_query(radius, nsample, xyz, new_xyz):
    B, N, _ = xyz.shape
    S = new_xyz.shape[1]
    ns = min(nsample, N)
    group = torch.arange(N, device=xyz.device).view(1,1,N).expand(B,S,N).clone()
    group[square_distance(new_xyz, xyz) > radius**2] = N
    group = group.sort(-1)[0][:, :, :ns]
    first = group[:, :, :1].expand(-1,-1,ns)
    group[group == N] = first[group == N]
    return group

class SAModule(nn.Module):
    def __init__(self, npt, r, ns, in_ch, mlp_chs):
        super().__init__()
        self.npt, self.r, self.ns = npt, r, ns
        layers, last = [], in_ch + 3
        for out in mlp_chs:
            layers += [nn.Conv2d(last, out, 1), nn.BatchNorm2d(out), nn.ReLU()]
            last = out
        self.mlp = nn.Sequential(*layers)
    def forward(self, xyz, feat):
        xy_t = xyz.permute(0,2,1)
        fps  = farthest_point_sample(xy_t, self.npt)
        new_xyz = index_points(xy_t, fps)
        idx  = ball_query(self.r, self.ns, xy_t, new_xyz)
        grouped = index_points(xy_t, idx) - new_xyz.unsqueeze(2)
        if feat is not None:
            grouped = torch.cat([grouped, index_points(feat.permute(0,2,1), idx)], -1)
        return new_xyz.permute(0,2,1), self.mlp(grouped.permute(0,3,2,1)).max(2)[0]

class FPModule(nn.Module):
    def __init__(self, in_ch, mlp_chs):
        super().__init__()
        layers, last = [], in_ch
        for out in mlp_chs:
            layers += [nn.Conv1d(last, out, 1), nn.BatchNorm1d(out), nn.ReLU()]
            last = out
        self.mlp = nn.Sequential(*layers)
    def forward(self, xyz1, xyz2, f1, f2):
        d = square_distance(xyz1.permute(0,2,1), xyz2.permute(0,2,1))
        d, idx = d.sort(-1)
        d, idx = d[:,:,:3].clamp(1e-10), idx[:,:,:3]
        w = (1.0/d); w = w / w.sum(-1, keepdim=True)
        interp = (index_points(f2.permute(0,2,1), idx) * w.unsqueeze(-1)).sum(2)
        out = interp.permute(0,2,1)
        if f1 is not None:
            out = torch.cat([f1, out], 1)
        return self.mlp(out)

class PointNet2Seg(nn.Module):
    def __init__(self, nc=NUM_CLASSES, in_ch=7):
        super().__init__()
        self.sa1 = SAModule(1024, 0.1, 32, in_ch,  [32, 32, 64])
        self.sa2 = SAModule(256,  0.2, 32, 64+3,   [64, 64, 128])
        self.sa3 = SAModule(64,   0.4, 32, 128+3,  [128,128,256])
        self.fp3 = FPModule(256+128, [256,256])
        self.fp2 = FPModule(256+64,  [256,128])
        self.fp1 = FPModule(128,     [128,128,128])
        self.head = nn.Sequential(nn.Conv1d(128,128,1), nn.BatchNorm1d(128),
                                  nn.ReLU(), nn.Dropout(0.4), nn.Conv1d(128,nc,1))
    def forward(self, x):
        l1x,l1f = self.sa1(x,x)
        l2x,l2f = self.sa2(l1x,l1f)
        l3x,l3f = self.sa3(l2x,l2f)
        l2f = self.fp3(l2x,l3x,l2f,l3f)
        l1f = self.fp2(l1x,l2x,l1f,l2f)
        f0  = self.fp1(x,l1x,None,l1f)
        return self.head(f0)

model = PointNet2Seg()
log.info(f"PointNet++ parameters: {sum(p.numel() for p in model.parameters()):,}")


## Dataset & DataLoader

In [ ]:
# ── Dataset: one random chunk per call ───────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

class PointCloudDataset(Dataset):
    """Returns one fixed-size sphere-crop chunk per item.
    This class only handles data — no training logic here."""
    def __init__(self, files, augment=True):
        self.files   = list(files)
        self.augment = augment
        self.N       = CONFIG["num_points"]
        self.chunks  = CONFIG["chunks_per_cloud"]

    def __len__(self):
        return len(self.files) * self.chunks

    def __getitem__(self, idx):
        path = self.files[idx // self.chunks]
        pts, feat, lbl = get_features(path)
        n = len(feat)

        # pick N nearest points around a random seed point
        if n <= self.N:
            chosen = np.random.choice(n, self.N, replace=True)
        else:
            seed = np.random.randint(n)
            dist = ((feat[:, :3] - feat[seed, :3]) ** 2).sum(1)
            chosen = np.argpartition(dist, self.N - 1)[:self.N]

        x = feat[chosen].copy()    # (N, 7)
        y = lbl[chosen].copy()     # (N,)

        # data augmentation: random rotation around vertical axis
        if self.augment:
            angle = np.random.uniform(0, 2 * np.pi)
            c, s  = np.cos(angle), np.sin(angle)
            R     = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], np.float32)
            x[:, :3]  = x[:, :3]  @ R.T   # rotate xyz
            x[:, 4:7] = x[:, 4:7] @ R.T   # rotate normals

        # shape: (channels, points) for Conv1d / attention layers
        return torch.from_numpy(x.T), torch.from_numpy(y)


## Training

In [ ]:
# ── Training loop with periodic checkpointing ────────────────────────────────
# Every SAVE_EVERY epochs:  saves model + optimizer state → resume after power cut
# Always keeps:             the BEST val-mIoU model as a separate file
# On resume:                loads the LATEST periodic checkpoint and continues
# Disk policy:              only keeps the last 2 periodic checkpoints (saves space)

SAVE_EVERY = 10    # save a checkpoint every this many epochs

def train_model(model, model_name):
    """Train the model with periodic saving. Resumes automatically if a
    checkpoint exists. Returns the model loaded with the best weights."""

    ckpt_dir  = CONFIG["checkpoint_dir"]
    best_path = os.path.join(ckpt_dir, f"{model_name}_best.pth")

    # ── helper: find the latest periodic checkpoint ───────────────────────────
    def latest_periodic():
        """Return (epoch, path) of the most recent periodic checkpoint, or (0, None)."""
        pattern = os.path.join(ckpt_dir, f"{model_name}_epoch_*.pth")
        files   = sorted(glob.glob(pattern))   # sorted alphabetically = epoch order
        if not files:
            return 0, None
        # extract epoch number from filename, pick the highest
        def epoch_of(p):
            try:   return int(os.path.basename(p).split("_epoch_")[1].replace(".pth",""))
            except: return 0
        files.sort(key=epoch_of)
        return epoch_of(files[-1]), files[-1]

    # ── helper: delete old periodic checkpoints, keep only the last 2 ─────────
    def prune_old_checkpoints(keep=2):
        pattern = os.path.join(ckpt_dir, f"{model_name}_epoch_*.pth")
        files   = sorted(glob.glob(pattern))
        def epoch_of(p):
            try:   return int(os.path.basename(p).split("_epoch_")[1].replace(".pth",""))
            except: return 0
        files.sort(key=epoch_of)
        for old in files[:-keep]:
            try:    os.remove(old); log.info(f"Removed old checkpoint: {old}")
            except: pass

    # ── data loaders ──────────────────────────────────────────────────────────
    train_loader = DataLoader(
        PointCloudDataset(TRAIN_FILES, augment=True),
        batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True)
    val_loader = DataLoader(
        PointCloudDataset(VAL_FILES, augment=False),
        batch_size=CONFIG["batch_size"], shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"],
                                  weight_decay=1e-4)
    loss_fn   = nn.CrossEntropyLoss()

    # ── try to resume from the latest periodic checkpoint ────────────────────
    start_epoch = 1
    best_miou   = -1.0
    no_improve  = 0
    history     = []

    resume_epoch, resume_path = latest_periodic()
    if resume_path:
        log.info(f"Resuming from {resume_path} (epoch {resume_epoch})")
        ckpt = torch.load(resume_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_epoch = ckpt["epoch"] + 1
        best_miou   = ckpt.get("best_miou",   -1.0)
        no_improve  = ckpt.get("no_improve",   0)
        history     = ckpt.get("history",      [])
        log.info(f"Resumed: start_epoch={start_epoch}  best_miou={best_miou:.4f}")
    else:
        log.info(f"No checkpoint found — training from scratch.")

    # ── if already fully trained, just load best and return ──────────────────
    if start_epoch > CONFIG["epochs"]:
        log.info("Training already complete. Loading best model.")
        state = torch.load(best_path, map_location="cpu", weights_only=False)
        model.load_state_dict(state["model_state"])
        return model

    model.to(DEVICE)

    run_name = f"{model_name}_seg"
    with (mlflow.start_run(run_name=run_name) if MLFLOW_OK
          else open(os.devnull, "w")) as _:

        if MLFLOW_OK:
            mlflow.log_params({
                "model"       : model_name,
                "epochs"      : CONFIG["epochs"],
                "batch_size"  : CONFIG["batch_size"],
                "num_points"  : CONFIG["num_points"],
                "lr"          : CONFIG["lr"],
                "start_epoch" : start_epoch,
            })

        for epoch in range(start_epoch, CONFIG["epochs"] + 1):

            # ── train one epoch ───────────────────────────────────────────────
            model.train()
            for x, y in train_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                loss_fn(model(x), y).backward()
                optimizer.step()

            # ── validate ─────────────────────────────────────────────────────
            model.eval()
            all_true, all_pred = [], []
            with torch.no_grad():
                for x, y in val_loader:
                    out = model(x.to(DEVICE)).argmax(dim=1).cpu().numpy()
                    all_pred.extend(out.ravel())
                    all_true.extend(y.numpy().ravel())

            miou = compute_miou(np.array(all_true), np.array(all_pred), NUM_CLASSES)
            history.append({"epoch": epoch, "val_miou": miou})
            log.info(f"Epoch {epoch:3d}/{CONFIG['epochs']} | val mIoU = {miou:.4f}")

            if MLFLOW_OK:
                mlflow.log_metric("val_miou", miou, step=epoch)

            # ── update best model ─────────────────────────────────────────────
            if miou > best_miou + 1e-4:
                best_miou  = miou
                no_improve = 0
                # overwrite best checkpoint
                torch.save({"model_state": model.state_dict(),
                            "best_val_miou": best_miou}, best_path)
                log.info(f"  ✓ New best mIoU {best_miou:.4f} → saved {best_path}")
            else:
                no_improve += 1

            # ── periodic checkpoint (every SAVE_EVERY epochs) ─────────────────
            if epoch % SAVE_EVERY == 0 or epoch == CONFIG["epochs"]:
                periodic_path = os.path.join(ckpt_dir,
                                             f"{model_name}_epoch_{epoch:04d}.pth")
                torch.save({
                    "model_state"     : model.state_dict(),
                    "optimizer_state" : optimizer.state_dict(),
                    "epoch"           : epoch,
                    "best_miou"       : best_miou,
                    "no_improve"      : no_improve,
                    "history"         : history,
                }, periodic_path)
                log.info(f"  💾 Periodic checkpoint saved → {periodic_path}")
                prune_old_checkpoints(keep=2)   # keep only last 2 periodic files

            # ── early stopping ────────────────────────────────────────────────
            if no_improve >= CONFIG["patience"]:
                log.info(f"Early stop at epoch {epoch} "
                         f"(no improvement for {CONFIG['patience']} epochs)")
                break

        if MLFLOW_OK:
            mlflow.log_metric("best_val_miou", best_miou)

    # ── load best weights before returning ────────────────────────────────────
    if os.path.exists(best_path):
        state = torch.load(best_path, map_location="cpu", weights_only=False)
        model.load_state_dict(state["model_state"])
        log.info(f"Loaded best model (mIoU={best_miou:.4f}) from {best_path}")
    model.to("cpu")
    return model


In [ ]:
model = train_model(model, MODEL_NAME)

## Visualization helper

In [ ]:
# ── Visualization: Open3D ────────────────────────────────────────────────────
# Green = target (wood powder)   |   Red = others (environment)
def visualize_segmentation(points, predictions, title="Segmentation"):
    """Open an Open3D window showing the segmentation result."""
    colors = np.zeros((len(points), 3), dtype=np.float64)
    colors[predictions == CONFIG["target_class"]] = [0.0, 0.8, 0.0]   # green
    colors[predictions != CONFIG["target_class"]] = [0.8, 0.0, 0.0]   # red

    pcd = o3d.geometry.PointCloud()
    # centre the cloud so it appears at the origin
    center = points.mean(axis=0)
    pcd.points = o3d.utility.Vector3dVector(
        (points - center).astype(np.float64))
    pcd.colors = o3d.utility.Vector3dVector(colors)

    # XYZ axes
    span = float((points.max(0) - points.min(0)).max()) * 0.5
    axes = o3d.geometry.TriangleMesh.create_coordinate_frame(size=span)

    n_target = int((predictions == CONFIG["target_class"]).sum())
    n_other  = len(predictions) - n_target
    full_title = (f"{title}  |  green(target)={n_target:,}  "
                  f"red(others)={n_other:,}")
    o3d.visualization.draw_geometries([pcd, axes],
                                       window_name=full_title,
                                       width=1280, height=800)


## Inference helper

In [ ]:
# ── Full-cloud inference ──────────────────────────────────────────────────────
@torch.no_grad()
def predict_full_cloud(model, path):
    """Label every point in a file using the trained model.
    The cloud is split into fixed-size chunks, each chunk goes through the model,
    results are stitched back together."""
    model.eval()
    pts, feat, lbl = get_features(path)
    n    = len(feat)
    N    = CONFIG["num_points"]

    # random order so every point is covered
    perm = np.random.RandomState(0).permutation(n)
    pad  = (N - n % N) % N
    if pad:
        perm = np.concatenate([perm, perm[:pad]])
    chunks = perm.reshape(-1, N)

    preds = np.zeros(n, dtype=np.int64)
    model.to(DEVICE)
    for start in range(0, len(chunks), CONFIG["batch_size"]):
        cid = chunks[start : start + CONFIG["batch_size"]]
        x   = torch.from_numpy(feat[cid].transpose(0, 2, 1)).to(DEVICE)
        out = model(x).argmax(dim=1).cpu().numpy()
        preds[cid.ravel()] = out.ravel()
    model.to("cpu")
    return pts, preds, lbl


## Testing  (held-out test files with labels)

In [ ]:
# ── Testing on held-out test files ───────────────────────────────────────────
log.info("=== TESTING ===")
test_mious = []
for path in TEST_FILES:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, lbl = predict_full_cloud(model, path)
    miou = compute_miou(lbl, preds, NUM_CLASSES)
    test_mious.append(miou)
    log.info(f"  {name}: mIoU = {miou:.4f}")

log.info(f"Mean test mIoU: {np.mean(test_mious):.4f}")

# ── Visualize the first few test files ───────────────────────────────────────
for path in TEST_FILES[:CONFIG["max_vis_files"]]:
    name = os.path.splitext(os.path.basename(path))[0]
    pts, preds, _ = predict_full_cloud(model, path)
    visualize_segmentation(pts, preds, title=f"{MODEL_NAME} | {name}")


## Final Inference  (`data/test`, no labels)

In [ ]:
# ── Final inference on data/test (no labels needed) ──────────────────────────
if INFER_FILES:
    log.info("=== FINAL INFERENCE ===")
    for path in INFER_FILES:
        name = os.path.splitext(os.path.basename(path))[0]
        pts, preds, _ = predict_full_cloud(model, path)
        n_tgt = int((preds == CONFIG["target_class"]).sum())
        log.info(f"  {name}: target={n_tgt:,} / total={len(preds):,}")
        visualize_segmentation(pts, preds, title=f"INFERENCE | {MODEL_NAME} | {name}")
else:
    log.info("data/test is empty — skipping inference.")
